# 11 — Frozen Freddie out-of-time reliability evaluation

**Purpose:** Score Freddie Mac 2018–2023 with the already frozen 2015–2017 Logistic Regression and XGBoost models. Twelve half-year origination cohorts are the primary unit; annual and quarterly views provide context and timing sensitivity. No refitting, recalibration, hyperparameter selection or OOT-derived preprocessing is permitted.

This is retrospective evaluation of the existing 24-month composite outcome. It does not assert that the complete 2015–2017 labels were available on 1 January 2018. Later steps study borrower-population drift, historical retraining and Fannie external validation separately.

Run locally after Notebook 10 has completed. The outputs are under `data/processed/Freddie_Mac/model_baselines_2015_2017/oot_evaluation/` and are excluded from Git.


In [1]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import joblib
from scipy.stats import norm
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

ROOT=Path.cwd().parent if Path.cwd().name.lower()=='notebooks' else Path.cwd()
assert (ROOT/'src'/'freddie_features.py').is_file()
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from src.freddie_features import (REQUIRED_SOURCE_COLUMNS,PREDICTOR_COLUMNS,
    build_engineered_features,apply_preprocessing_contract)

DATA=ROOT/'data'/'processed'/'Freddie_Mac'
INPUT=DATA/'freddie_modeling_dataset_2015_2023_24m.parquet'
BASE=DATA/'model_baselines_2015_2017'
OUT=BASE/'oot_evaluation'
MANIFEST=BASE/'model_manifest.json'
CONTRACT=BASE/'freddie_feature_contract.json'
assert INPUT.is_file() and MANIFEST.is_file() and CONTRACT.is_file(), 'Run Notebook 10 first.'
manifest=json.loads(MANIFEST.read_text(encoding='utf-8'))
contract=json.loads(CONTRACT.read_text(encoding='utf-8'))
def sha(path):
    with path.open('rb') as f:return hashlib.file_digest(f,'sha256').hexdigest()
assert sha(INPUT)==manifest['input_file_sha256'], 'Modeling dataset changed since baseline fit.'
assert sha(CONTRACT)==manifest['feature_contract_sha256'], 'Frozen feature contract changed.'
assert manifest['predictors']==PREDICTOR_COLUMNS and manifest['horizon_months']==24
assert manifest['development_years']==[2015,2016,2017]
models={}
for family in ('xgboost','logistic'):
    model_path=BASE/manifest['models'][family]['artifact']
    assert sha(model_path)==manifest['models'][family]['sha256'], f'{family} model changed.'
    models[family]=joblib.load(model_path)
OUT.mkdir(parents=True,exist_ok=True)
print('Verified frozen input, contract and model hashes.')


Verified frozen input, contract and model hashes.


## 1. Build evaluation predictors without learning from OOT cohorts

Use Notebook 07's origination-time feature definitions, then apply the frozen 2015–2017 contract. Keep loan IDs and cohort fields for audit only. No identifier, period, target or follow-up field enters `PREDICTOR_COLUMNS`. Score the 2015–2017 development loans solely to fix each model's calibration-bin edges; these predictions do not tune either model.


In [3]:
raw=pd.read_parquet(INPUT,columns=list(dict.fromkeys([*REQUIRED_SOURCE_COLUMNS,'HORIZON_MONTHS'])))
assert not raw.duplicated(['SOURCE YEAR','LOAN IDENTIFIER']).any()
assert set(raw['SOURCE YEAR'].astype(int))==set(range(2015,2024))
assert set(pd.to_numeric(raw['HORIZON_MONTHS']).unique())=={24}
engineered=build_engineered_features(raw)
del raw
feature_data=apply_preprocessing_contract(engineered,contract)
assert not feature_data[PREDICTOR_COLUMNS].isna().any().any()
dev=feature_data['SOURCE YEAR'].isin([2015,2016,2017]).to_numpy()
oot=feature_data['SOURCE YEAR'].isin([2018,2019,2020,2021,2022,2023]).to_numpy()
assert int(dev.sum())==manifest['development_rows']
assert int(feature_data.loc[dev,'TARGET_COMPOSITE_CREDIT_EVENT'].sum())==manifest['development_events']
assert dev.sum()+oot.sum()==len(feature_data)

pred=feature_data.loc[oot,['SOURCE YEAR','LOAN IDENTIFIER','ORIGINATION YEAR',
                             'ORIGINATION QUARTER','ORIGINATION COHORT',
                             'TARGET_COMPOSITE_CREDIT_EVENT']].copy()
pred=pred.rename(columns={'TARGET_COMPOSITE_CREDIT_EVENT':'y_true'})
pred['year']=pd.to_numeric(pred['ORIGINATION YEAR'],errors='raise').astype(int)
pred['quarter'] = pd.to_numeric(
    pred['ORIGINATION QUARTER'].astype('string').str.strip().str.removeprefix('Q'),
    errors='raise'
).astype(int)
assert pred['year'].between(2018,2023).all() and pred['quarter'].between(1,4).all()
assert (pred['SOURCE YEAR'].astype(int)==pred['year']).all()
pred['half']=np.where(pred['quarter']<=2,'H1','H2')
pred['half_year']=pred['year'].astype(str)+'-'+pred['half']
assert set(pred['y_true'].astype(int).unique())=={0,1}
bin_edges={}
for family,model in models.items():
    p_dev=model.predict_proba(feature_data.loc[dev,PREDICTOR_COLUMNS])[:,1]
    p_oot=model.predict_proba(feature_data.loc[oot,PREDICTOR_COLUMNS])[:,1]
    assert np.isfinite(p_dev).all() and np.isfinite(p_oot).all()
    assert ((p_oot>=0)&(p_oot<=1)).all()
    pred[f'p_{family}']=p_oot
    # Quantile edges are learned only from final development-model predictions.
    inner=np.unique(np.quantile(p_dev,[.2,.4,.6,.8])).tolist()
    bin_edges[family]=[-float('inf'),*inner,float('inf')]
assert len(pred)==int(oot.sum()) and not pred.duplicated(['SOURCE YEAR','LOAN IDENTIFIER']).any()
pred.to_parquet(OUT/'oot_predictions_2018_2023.parquet',index=False)
(OUT/'frozen_calibration_bins.json').write_text(
    json.dumps({family:edges[1:-1] for family,edges in bin_edges.items()},indent=2),
    encoding='utf-8')
print('OOT scored:',len(pred),'events:',int(pred['y_true'].sum()))
print('Half-year cohorts:',pred.groupby('half_year',sort=True).size().to_dict())


OOT scored: 294070 events: 6481
Half-year cohorts: {'2018-H1': 23699, '2018-H2': 23718, '2019-H1': 24326, '2019-H2': 24285, '2020-H1': 24841, '2020-H2': 24824, '2021-H1': 24714, '2021-H2': 24776, '2022-H1': 24633, '2022-H2': 24753, '2023-H1': 24757, '2023-H2': 24744}


## 2. Cohort metrics and uncertainty

Use scikit-learn **average precision** (AP), ROC-AUC, Brier score and observed-minus-mean-predicted risk. Report each cohort's count, positives, negatives and prevalence with a 95% Wilson interval. For each half-year, use 1,000 loan-level percentile bootstrap draws, seed 42. Use the **same indices for both models** to support paired comparisons. Discard and count a draw without both classes for AP/ROC-AUC. Intervals describe uncertainty within each cohort; they are not a test of causal drift.


In [4]:
def wilson(k,n,z=norm.ppf(.975)):
    if n==0:return (float('nan'),float('nan'))
    p=k/n; denominator=1+z*z/n
    center=(p+z*z/(2*n))/denominator
    half=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/denominator
    return (float(max(0,center-half)),float(min(1,center+half)))

def score(y,p):
    result={'brier':float(brier_score_loss(y,p)),
            'risk_gap':float(np.mean(y)-np.mean(p)),
            'mean_predicted_risk':float(np.mean(p))}
    result['roc_auc']=float(roc_auc_score(y,p)) if np.unique(y).size==2 else float('nan')
    result['average_precision']=float(average_precision_score(y,p)) if np.unique(y).size==2 else float('nan')
    return result

def row_metrics(period_kind,period,frame):
    y=frame['y_true'].to_numpy(dtype=int);n=len(y);k=int(y.sum())
    lo,hi=wilson(k,n)
    common={'period_kind':period_kind,'period':period,'n':n,'positives':k,
            'negatives':n-k,'prevalence':k/n,'prevalence_ci_low':lo,'prevalence_ci_high':hi}
    return [{**common,'model':family,**score(y,frame[f'p_{family}'].to_numpy(dtype=float))}
            for family in ('xgboost','logistic')]

expected=[f'{y}-{h}' for y in range(2018,2024) for h in ('H1','H2')]
assert sorted(pred['half_year'].unique())==sorted(expected)
rows=[]
for period,frame in pred.groupby('half_year',sort=True):rows.extend(row_metrics('half_year',period,frame))
for year,frame in pred.groupby('year',sort=True):rows.extend(row_metrics('annual',str(year),frame))
for (year,q),frame in pred.groupby(['year','quarter'],sort=True):
    rows.extend(row_metrics('quarter',f'{year}-Q{q}',frame))
summary=pd.DataFrame(rows)
assert len(summary)==(12+6+24)*2
assert int(summary.loc[(summary.period_kind=='half_year')&(summary.model=='xgboost'),'n'].sum())==len(pred)
assert int(summary.loc[(summary.period_kind=='half_year')&(summary.model=='xgboost'),'positives'].sum())==int(pred.y_true.sum())

metric_names=('roc_auc','average_precision','brier','risk_gap')
boot_rows=[]; rng=np.random.default_rng(42)
for period in expected:
    frame=pred.loc[pred.half_year==period]
    y=frame.y_true.to_numpy(dtype=int)
    p={family:frame[f'p_{family}'].to_numpy(dtype=float) for family in ('xgboost','logistic')}
    draws={family:{m:[] for m in metric_names} for family in p}
    differences={m:[] for m in metric_names}
    invalid_rank_draws=0
    for _ in range(1000):
        ix=rng.integers(0,len(y),size=len(y));yb=y[ix]
        valid=np.unique(yb).size==2
        if not valid:invalid_rank_draws+=1
        values={family:score(yb,probs[ix]) for family,probs in p.items()}
        for family in p:
            for m in metric_names:
                if valid or m in ('brier','risk_gap'):draws[family][m].append(values[family][m])
        for m in metric_names:
            if valid or m in ('brier','risk_gap'):
                differences[m].append(values['xgboost'][m]-values['logistic'][m])
    for family in p:
        for m in metric_names:
            vals=draws[family][m]
            low,high=np.quantile(vals,[.025,.975]) if vals else (float('nan'),float('nan'))
            boot_rows.append({'period':period,'comparison':family,'metric':m,
                              'ci_low':float(low),'ci_high':float(high),
                              'valid_draws':len(vals),'invalid_rank_draws':invalid_rank_draws})
    for m in metric_names:
        vals=differences[m]
        low,high=np.quantile(vals,[.025,.975]) if vals else (float('nan'),float('nan'))
        boot_rows.append({'period':period,'comparison':'xgboost_minus_logistic','metric':m,
                          'ci_low':float(low),'ci_high':float(high),
                          'valid_draws':len(vals),'invalid_rank_draws':invalid_rank_draws})
    print('Bootstrap:',period,'invalid ranking draws:',invalid_rank_draws)
boot=pd.DataFrame(boot_rows)
summary.to_csv(OUT/'cohort_reliability_summary.csv',index=False)
boot.to_csv(OUT/'half_year_bootstrap_intervals.csv',index=False)
display(summary.query("period_kind=='half_year'").round(5))


Bootstrap: 2018-H1 invalid ranking draws: 0
Bootstrap: 2018-H2 invalid ranking draws: 0
Bootstrap: 2019-H1 invalid ranking draws: 0
Bootstrap: 2019-H2 invalid ranking draws: 0
Bootstrap: 2020-H1 invalid ranking draws: 0
Bootstrap: 2020-H2 invalid ranking draws: 0
Bootstrap: 2021-H1 invalid ranking draws: 0
Bootstrap: 2021-H2 invalid ranking draws: 0
Bootstrap: 2022-H1 invalid ranking draws: 0
Bootstrap: 2022-H2 invalid ranking draws: 0
Bootstrap: 2023-H1 invalid ranking draws: 0
Bootstrap: 2023-H2 invalid ranking draws: 0


,period_kind,period,n,positives,negatives,prevalence,prevalence_ci_low,prevalence_ci_high,model,brier,risk_gap,mean_predicted_risk,roc_auc,average_precision
0,half_year,2018-H1,23699,327,23372,0.01380,0.01239,0.01536,xgboost,0.01347,0.00466,0.00914,0.74321,0.04441
1,half_year,2018-H1,23699,327,23372,0.01380,0.01239,0.01536,logistic,0.01352,0.00385,0.00995,0.74590,0.04189
2,half_year,2018-H2,23718,895,22823,0.03774,0.03538,0.04024,xgboost,0.03647,0.02761,0.01012,0.70929,0.08943
3,half_year,2018-H2,23718,895,22823,0.03774,0.03538,0.04024,logistic,0.03626,0.02608,0.01166,0.72054,0.09568
4,half_year,2019-H1,24326,1106,23220,0.04547,0.04292,0.04816,xgboost,0.04413,0.03668,0.00878,0.70269,0.09810
5,half_year,2019-H1,24326,1106,23220,0.04547,0.04292,0.04816,logistic,0.04407,0.03600,0.00946,0.70845,0.09827
6,half_year,2019-H2,24285,1121,23164,0.04616,0.04359,0.04887,xgboost,0.04517,0.04048,0.00568,0.70034,0.10298
7,half_year,2019-H2,24285,1121,23164,0.04616,0.04359,0.04887,logistic,0.04515,0.04020,0.00596,0.70476,0.10296
8,half_year,2020-H1,24841,607,24234,0.02444,0.02259,0.02643,xgboost,0.02406,0.02032,0.00411,0.71756,0.05831
9,half_year,2020-H1,24841,607,24234,0.02444,0.02259,0.02643,logistic,0.02404,0.02024,0.00419,0.73001,0.06282


## 3. Frozen-bin calibration detail

Apply each model's development-defined quintile edges unchanged to all half-year cohorts. Duplicate development quantiles were merged when edges were saved. Report each nonempty bin's loans, events, average predicted risk, observed event rate and Wilson interval. Empty bins are explicit. Brier is a combined probability score, so calibration is also shown separately through risk gap and these bin tables. ECE is not a primary result.


In [5]:
cal=[]
for period in expected:
    frame=pred.loc[pred.half_year==period]
    for family in ('xgboost','logistic'):
        edges=np.asarray(bin_edges[family],dtype=float)
        which=np.searchsorted(edges[1:-1],frame[f'p_{family}'].to_numpy(dtype=float),side='right')
        for j in range(len(edges)-1):
            subset=frame.loc[which==j]
            n=len(subset);k=int(subset.y_true.sum()) if n else 0
            lo,hi=wilson(k,n)
            cal.append({'period':period,'model':family,'bin_index':j+1,
                        'lower_edge':edges[j],'upper_edge':edges[j+1],
                        'n':n,'events':k,'mean_predicted':float(subset[f'p_{family}'].mean()) if n else np.nan,
                        'observed_rate':k/n if n else np.nan,
                        'wilson_low':lo,'wilson_high':hi})
calibration=pd.DataFrame(cal)
for (period,family),part in calibration.groupby(['period','model']):
    cohort=pred.loc[pred.half_year==period]
    assert part.n.sum()==len(cohort) and part.events.sum()==int(cohort.y_true.sum())
calibration.to_csv(OUT/'half_year_calibration_bins.csv',index=False)
print('Calibration rows:',len(calibration),'empty bins:',int((calibration.n==0).sum()))
display(calibration.head(10).round(5))


Calibration rows: 120 empty bins: 0


,period,model,bin_index,lower_edge,upper_edge,n,events,mean_predicted,observed_rate,wilson_low,wilson_high
0,2018-H1,xgboost,1,-inf,0.00083,2374,2,0.00059,0.00084,0.00023,0.00307
1,2018-H1,xgboost,2,0.00083,0.00169,3932,19,0.00123,0.00483,0.00310,0.00754
2,2018-H1,xgboost,3,0.00169,0.00347,4873,32,0.00248,0.00657,0.00466,0.00926
3,2018-H1,xgboost,4,0.00347,0.00865,5592,58,0.00564,0.01037,0.00803,0.01338
4,2018-H1,xgboost,5,0.00865,inf,6928,216,0.02407,0.03118,0.02734,0.03554
5,2018-H1,logistic,1,-inf,0.00102,2678,7,0.00067,0.00261,0.00127,0.00539
6,2018-H1,logistic,2,0.00102,0.00207,3753,16,0.00152,0.00426,0.00263,0.00691
7,2018-H1,logistic,3,0.00207,0.00400,4638,26,0.00295,0.00561,0.00383,0.00820
8,2018-H1,logistic,4,0.00400,0.00862,5317,50,0.00598,0.00940,0.00714,0.01238
9,2018-H1,logistic,5,0.00862,inf,7313,228,0.02499,0.03118,0.02743,0.03541


## Review and interpretation gate

Upload these **small** outputs: `cohort_reliability_summary.csv`, `half_year_bootstrap_intervals.csv`, `half_year_calibration_bins.csv`, `frozen_calibration_bins.json`. Keep `oot_predictions_2018_2023.parquet` local for later paired drift analyses and audit. Do not merge the PR or infer a retraining need until these descriptive results have been checked. Low-event quarters require particular caution. This notebook does not evaluate population drift or Fannie Mae.
